In [ ]:
import os
import polars as pl
import json
from tqdm import tqdm

from etytreealg.decode import langcodes as langcodes

def load_df():
    df_dest = '../data/parquet/ety_expanded.parquet'
    return pl.read_parquet(df_dest)

def hydrate_df(df):
    """
    Call json.loads on the columns that are stored as JSON strings
    which are templates, related, and descendants
    """

    # note that Object datatype cannot be written to parquet/arrow
    df = df.with_columns([
        pl.col('templates').map_elements(lambda x: json.loads(x) if x else [], return_dtype=pl.Object).alias('templates_h'),
        pl.col('related').map_elements(lambda x: json.loads(x) if x else [], return_dtype=pl.Object).alias('related_h'),
        pl.col('descendants').map_elements(lambda x: json.loads(x) if x else [], return_dtype=pl.Object).alias('descendants_h'),
    ])

    df = df.drop(['templates', 'related', 'descendants'])

    print("Hydrated!")
    return df
def collate_templates(df, desired_templates):
    """
    collate the templates by their type
    """
    collated_templates = {x: [] for x in desired_templates}
    for templates in df['templates_h']:
        if not templates:
            continue
        for template in templates:
            if template['name'] in desired_templates:
                collated_templates[template['name']].append(template['args'])
    return collated_templates

def lang_sr_by_template_arg(args):
    """
    get empirical rate that a template argument is a langcode
    """
    from etytreealg.decode import langcodes as langcodes
    sr_by_arg = {} # success rate by key
    for arg in args:
        for key, value in arg.items():
            if key not in sr_by_arg:
                sr_by_arg[key] = {'success': 0, 'total': 0}
            if value:
                sr_by_arg[key]['total'] += 1
                if langcodes.is_langcode(value):
                    sr_by_arg[key]['success'] += 1
    return sr_by_arg



def predicted_lang_args_per_template(lang_success_rates, debug=False):
    """
    Empirically determine which arguments are likely to be lang codes in a given template.
    Does this based on the success rate of a key being a lang code.
    Returns a dictionary of template: str -> set[str]

    Criteria for being a suspected lang key
    1. key contains 'lang' OR
    2. success rate > 95% AND total > 5

    If debug is True, then we also return the warning keys, where the success rate is between 0.8 and 0.95.

    We have exceptions: 

    1. Han compound's 'ls' is one of ['i', 'ic', 'psc'] but do NOT represent langcodes
    2. Same for liushu: ['p', 'i', 'ic', 'psc']

    4. surface analysis & surf 1 can be a lang, but surf 1 can also be a reference to another template
    (like +suf)
    5. noncog contains a large frequency of language families, due to its migration from {{etyl}}
    6. Hira-dakuten 2 is actually Hepburn romanization of the character, which happens to be a lot of used 2- or 3- letter codes
    7. Kana-dakuten: same
    8. wp & zh-wp's argument is technically a lang code, but not the exact same one https://en.wiktionary.org/wiki/Template:wikipedia
    9. cog 1 may be a comma-separated list of langcodes. The same is true of bor 2, bor+ 2, 
    (1 instance only:) clq 2, cal 2, ubor 2, ncog 1,
    10. dercat may have "<" in fields normally occupied by a language.
    11. inc-ext looks confusing
    12. pi-root 1 and tl-bay sc are NOT languages

    """
    suspected_lang_keys_per_template = {}
    _warning_keys = {}
    for template, sr in lang_success_rates.items():
        # criteria for being a suspected lang key
        # 1. key contains 'lang' OR
        # 2. success rate > 95% AND total > 5
        suspected_lang_keys = {k for k, v in sr.items() if 'lang' in k or v['total'] and v['total'] >= 5 and v['success'] / v['total'] >= 0.95}
        
        # check for those between 0.8 and 0.95 and print them as a warning: usually, 
        for k, v in sr.items():
            if v['total'] and v['total'] >= 5 and 0.5 <= v['success'] / v['total'] < .9999:
                # print(f"WARNING: {template} {k} {v['success'] / v['total']:.2%}")
                _warning_keys[(template, k)] = v['success'] / v['total']
        
        suspected_lang_keys_per_template[template] = list(suspected_lang_keys)
    if debug:
        return suspected_lang_keys_per_template, _warning_keys
    return suspected_lang_keys_per_template

In [41]:
df = load_df()
df = df.with_row_index("index")

In [65]:
df = hydrate_df(df)

Hydrated!


In [16]:
df

index,word,lang,lang_code,ety,etymology_number,num_templates,forms_of,templates_h,related_h,descendants_h
u32,str,str,str,str,u8,u32,list[str],object,object,object
0,"""""","""""","""""",null,0,0,null,null,null,null
1,"""apples and pears""","""English""","""en""",null,0,0,null,null,null,null
2,"""abhal""","""English""","""en""","""From Arabic أَبْهَل (ʔabhal).""",0,1,null,"[{'name': 'der', 'args': {'1': 'en', '2': 'ar', '3': 'أَبْهَل'}, 'expansion': 'Arabic أَبْهَل (ʔabhal)'}]",null,null
3,"""abaisance""","""English""","""en""","""From Old French abaissance.""",0,1,null,"[{'name': 'der', 'args': {'1': 'en', '2': 'fro', '3': 'abaissance'}, 'expansion': 'Old French abaissance'}]",null,null
4,"""A 1""","""English""","""en""",null,0,0,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…
9633550,"""copulate with""","""English""","""en""",null,0,0,null,null,null,null
9633551,"""physical property""","""English""","""en""",null,0,0,null,null,null,null
9633552,"""unit of measure""","""English""","""en""",null,0,0,null,null,null,null


In [66]:
with open('../data/templates/template_freqs.json', 'r') as f:
    sorted_template_frequencies = json.load(f)

In [67]:
desired_templates = [template for template, freq in sorted_template_frequencies if freq >= 20]

collated_templates = collate_templates(df, desired_templates)

In [8]:
collated_templates['der']

[{'1': 'en', '2': 'ar', '3': 'أَبْهَل'},
 {'1': 'en', '2': 'fro', '3': 'abaissance'},
 {'1': 'en', '2': 'la', '3': 'fabaceus'},
 {'1': 'fr', '2': 'fro', '3': 'abisme'},
 {'1': 'fr', '2': 'VL.', '3': '*abyssimus'},
 {'1': 'fr', '2': 'grc', '3': 'ἄβυσσος'},
 {'1': 'mfe', '2': 'fr', '3': 'abîmer'},
 {'1': 'en', '2': 'la', '3': 'abāctiō'},
 {'1': 'en', '2': 'la', '3': '-'},
 {'1': 'enm',
  '2': 'fro',
  '3': 'abaubir',
  '4': '',
  '5': 'to frighten, disconcert'},
 {'1': 'enm', '2': 'la', '3': 'ad-'},
 {'1': 'en', '2': 'la', '3': 'abies', '4': '', '5': 'silver fir (tree)'},
 {'1': 'en', '2': 'la', '3': 'ab'},
 {'1': 'en',
  '2': 'la',
  '3': 'abditīvus',
  '4': '',
  '5': 'removed or separated from'},
 {'1': 'en', '2': 'la', '3': 'abalienatio'},
 {'1': 'oc', '2': 'la', '3': 'grātuītus'},
 {'1': 'ro', '2': 'la', '3': 'gratuitus'},
 {'1': 'en', '2': 'la', '3': 'ab', 't': 'from'},
 {'1': 'en', '2': 'la', '3': 'abiudicare'},
 {'1': 'en', '2': 'la', '3': 'abiugātus'},
 {'1': 'en', '2': 'NL.', '

In [ ]:
lang_success_rates = {tname: lang_sr_by_template_arg(t_examples) for tname, t_examples in collated_templates.items()}

In [ ]:
lang_success_rates['der']

{'1': {'success': 276399, 'total': 276399},
 '2': {'success': 276390, 'total': 276399},
 '3': {'success': 2209, 'total': 268594},
 '4': {'success': 9, 'total': 9591},
 '5': {'success': 1106, 'total': 46334},
 't': {'success': 553, 'total': 26591},
 'tr': {'success': 228, 'total': 11233},
 'sc': {'success': 0, 'total': 554},
 'lit': {'success': 2, 'total': 1475},
 'pos': {'success': 18, 'total': 1479},
 'sort': {'success': 0, 'total': 2088},
 'ts': {'success': 18, 'total': 1002},
 'id': {'success': 28, 'total': 302},
 'g': {'success': 0, 'total': 415},
 'g2': {'success': 0, 'total': 0},
 'g3': {'success': 0, 'total': 0},
 'nocat': {'success': 0, 'total': 115},
 'gloss': {'success': 15, 'total': 846},
 'alt': {'success': 0, 'total': 135}}

In [24]:
all_words_set = df['word'].unique()

In [26]:
'hola' in all_words_set

True

In [27]:
type(all_words_set)

polars.series.series.Series

In [28]:
def word_sr_by_template_arg(args: list[dict[str, str]], known_lang_keys: list[str], all_words_set: pl.Series):
    """
    get empirical rate that a template argument is a word

    criteria for being a suspected word key:
    1. key is not a known LANG key
    
    metrics:
    1. word success rate: the value is present in all_words_set
    2. nonlatin rate: the value has at least one non-latin character


    """
    sr_by_arg = {} # word success rate by key
    for arg in args:
        for key, value in arg.items():
            if key in known_lang_keys:
                continue
            if key not in sr_by_arg:
                sr_by_arg[key] = {'total': 0, 'success': 0, 'nonlatin': 0}
            if value:
                sr_by_arg[key]['total'] += 1
                if value in all_words_set:
                    sr_by_arg[key]['success'] += 1
                if not value.isascii():
                    sr_by_arg[key]['nonlatin'] += 1
                
    return sr_by_arg

In [36]:
suspected_lang_keys_per_template, _warning_keys = predicted_lang_args_per_template(lang_success_rates, debug=True)

In [39]:
all_words_set = set(all_words_set)

In [41]:
'hola' in all_words_set

True

In [46]:
df.filter(pl.col('lang').str.starts_with('Proto-'))

index,word,lang,lang_code,ety,templates,etymology_number,num_templates,related,descendants,forms_of
u32,str,str,str,str,str,u8,u32,str,str,list[str]
94,"""frijaz""","""Proto-Germanic""","""gem-pro""","""From Proto-Indo-European *priH…","""[{""name"": ""root"", ""args"": {""1""…",0,2,"""[{""word"": ""frija\u00few\u014d""…","""[{""depth"": 1, ""templates"": [{""…",null
3819,"""wage""","""Proto-Norse""","""gmq-pro""",null,null,0,0,null,null,null
12323,"""after""","""Proto-Norse""","""gmq-pro""",null,null,0,0,null,null,null
13303,"""was""","""Proto-Norse""","""gmq-pro""",null,null,0,0,null,null,null
21794,"""an""","""Proto-Norse""","""gmq-pro""",null,null,0,0,null,null,null
…,…,…,…,…,…,…,…,…,…,…
9632368,"""leuþōn""","""Proto-West Germanic""","""gmw-pro""","""From Proto-Germanic *leuþōną. …","""[{""name"": ""inh"", ""args"": {""1"":…",0,2,null,"""[{""depth"": 1, ""templates"": [{""…",null
9632457,"""frakwistijan""","""Proto-West Germanic""","""gmw-pro""","""From Proto-Germanic *frakwisti…","""[{""name"": ""inh"", ""args"": {""1"":…",0,2,null,"""[{""depth"": 1, ""templates"": [{""…",null
9632478,"""frakwistijaną""","""Proto-Germanic""","""gem-pro""","""From *fra- + *kwistijaną.""","""[{""name"": ""af"", ""args"": {""1"": …",0,1,null,"""[{""depth"": 1, ""templates"": [{""…",null


In [42]:


# word_success_rates = {tname: word_sr_by_template_arg(t_examples, suspected_lang_keys_per_template.get(tname, []), all_words_set) for tname, t_examples in collated_templates.items()}


word_success_rates = {}
for tname, t_examples in tqdm(collated_templates.items()):
    suspected_lang_keys = suspected_lang_keys_per_template.get(tname, [])
    word_success_rates[tname] = word_sr_by_template_arg(t_examples, suspected_lang_keys, all_words_set)

100%|██████████| 331/331 [00:04<00:00, 71.89it/s] 


In [48]:
word_success_rates['root']

{'3': {'total': 33476, 'success': 28528, 'nonlatin': 26764, 'unique': 3286},
 'id1': {'total': 106, 'success': 106, 'nonlatin': 0, 'unique': 32},
 '4': {'total': 1440, 'success': 1244, 'nonlatin': 1115, 'unique': 472},
 'id': {'total': 2682, 'success': 2672, 'nonlatin': 0, 'unique': 120},
 '5': {'total': 136, 'success': 125, 'nonlatin': 105, 'unique': 82},
 'id2': {'total': 114, 'success': 114, 'nonlatin': 0, 'unique': 37},
 'id3': {'total': 6, 'success': 6, 'nonlatin': 0, 'unique': 5},
 '6': {'total': 18, 'success': 14, 'nonlatin': 11, 'unique': 14},
 '7': {'total': 3, 'success': 3, 'nonlatin': 3, 'unique': 2},
 'sort': {'total': 1, 'success': 0, 'nonlatin': 0, 'unique': 1}}

In [35]:
import json

with open('../data/templates/word_sr.json', 'r') as f:
    word_success_rates = json.load(f)

In [60]:
def predicted_word_keys_per_template(word_success_rates, debug=False):
    """
    Criteria for being a WORD

    1. filter to only allow keys that are strictly numerical
    2. preference is given to the most common key
    3. preference is given to the key, strictly numerical, with the lowest value

    4. require that key has success rate > 30%
    5. preference is given if key has success rate > 80%
    """
    predicted_word_keys = {}
    nonlexical = set()

    def sr(v):
        return v['success'] / v['total'] if v['total'] else 0
    def unique_r(v):
        return v['unique'] / v['total'] if v['total'] else 0
    for template, srs_full in word_success_rates.items():
        # criteria for being a suspected lang key
        # 1. key contains 'lang' OR
        # 2. success rate > 95% AND total > 5
        suspected = set()
        
        success_rates = {k: v for k, v in srs_full.items() if k.isnumeric()}

        if not success_rates:
            predicted_word_keys[template] = []
            continue
        lowest_k = min(success_rates, key=lambda x: int(x))
        most_freq_k = max(success_rates, key=lambda x: success_rates[x]['total'])

        # give preference to these candidates
        if sr(success_rates[lowest_k]) > 0.3:
            suspected.add(lowest_k)
        if sr(success_rates[most_freq_k]) > 0.3:
            suspected.add(most_freq_k)

        # for the non-preferential, check for those between 0.3 and 0.8
        for k, v in success_rates.items():
            if sr(v) > 0.7 and unique_r(v) > 0.1:
                suspected.add(k)
                # print(f"WARNING: {template} {k} {v['success'] / v['total']:.2%}")
                pass
        
        if not suspected:
            # there are numerical keys, but none of them are words.
            # this is unusual, and should be noted
            nonlexical.add(template)
            

        
        predicted_word_keys[template] = sorted(suspected, key=lambda x: int(x))
    if debug:
        return predicted_word_keys, nonlexical
    return predicted_word_keys

In [61]:
predicted_word_keys, nonlexical = predicted_word_keys_per_template(word_success_rates, debug=True)

In [50]:
predicted_word_keys['cog']

['2']

In [62]:
multilexical = {k: v for k, v in predicted_word_keys.items() if len(v) > 1}

In [63]:
multilexical

{'af': ['2', '3', '4', '5', '6', '7', '9', '10'],
 'suffix': ['2', '3', '4', '5', '6'],
 'affix': ['2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12'],
 'prefix': ['2', '3', '4', '5', '6', '7'],
 'compound': ['2',
  '3',
  '4',
  '5',
  '6',
  '7',
  '8',
  '9',
  '10',
  '11',
  '12',
  '13',
  '14',
  '15',
  '16',
  '17'],
 'root': ['3', '4', '5', '6', '7'],
 'com': ['2', '3', '4', '5', '6', '7', '8', '10', '17', '20', '22', '28'],
 'doublet': ['2',
  '3',
  '4',
  '5',
  '6',
  '7',
  '8',
  '9',
  '10',
  '11',
  '12',
  '13',
  '14',
  '15',
  '16',
  '17',
  '18',
  '19',
  '20',
  '21'],
 'suf': ['2', '3', '4', '5'],
 'surf': ['2', '3', '4', '5', '6'],
 'confix': ['2', '3', '4'],
 'ko-etym-sino': ['1', '3', '4', '5', '6', '7', '8'],
 'vi-etym-sino': ['1', '3', '5', '7', '8'],
 'blend': ['2', '3', '4'],
 'pre': ['2', '3', '4'],
 'liushu': ['1', '2', '3'],
 'Han compound': ['1', '2', '3', '4'],
 'com+': ['2', '3', '4', '5', '6', '7'],
 'ja-compound': ['1', '2', '3', '4', '5

In [70]:
word_success_rates['adverbial accusative']

{'nocap': {'total': 18, 'success': 18, 'nonlatin': 0, 'unique': 2},
 '2': {'total': 87, 'success': 14, 'nonlatin': 81, 'unique': 82},
 'gloss': {'total': 11, 'success': 7, 'nonlatin': 0, 'unique': 10},
 '3': {'total': 5, 'success': 3, 'nonlatin': 5, 'unique': 5},
 '4': {'total': 20, 'success': 11, 'nonlatin': 0, 'unique': 19},
 't': {'total': 24, 'success': 20, 'nonlatin': 0, 'unique': 24},
 'tr': {'total': 4, 'success': 3, 'nonlatin': 4, 'unique': 4},
 'nocat': {'total': 1, 'success': 1, 'nonlatin': 0, 'unique': 1}}

In [72]:
collated_templates['cardinalbox']

[{'1': 'ja', '2': '9', '3': '10', '4': '11', '5': '九', '6': '十一'},
 {'1': 'ja', '2': '9', '3': '10', '4': '11', '5': '九', '6': '十一'},
 {'1': 'ja', '2': '9', '3': '10', '4': '11', '5': '九', '6': '十一'},
 {'1': 'vi',
  '2': '2',
  '3': '3',
  '4': '4',
  '5': 'hai',
  '6': 'bốn',
  'ord': 'thứ ba'},
 {'1': 'kbd',
  '2': '7',
  '3': '8',
  '4': '9',
  '5': 'блы',
  '6': 'бгъу',
  'ord': ''},
 {'1': 'zom', '2': '1', '3': '2', '4': '3', '5': 'khet', '6': 'thum'},
 {'1': 'bi', '2': '2', '3': '3', '4': '4', '5': 'tu', '6': 'fo'},
 {'1': 'lou', '2': '5', '3': '6', '4': '7', '5': 'sink', '6': 'sèt'},
 {'1': 'id', '2': '3', '3': '4', '4': '5', '5': 'tiga', '6': 'lima'},
 {'1': 'tkl', '2': '5', '3': '6', '4': '7', '5': 'lima', '6': 'fitu'},
 {'1': 'tkl', '2': '1', '3': '2', '4': '3', '5': 'tahi', '6': 'tolu'},
 {'1': 'tkl', '2': '4', '3': '5', '4': '6', '5': 'fā', '6': 'ono'},
 {'1': 'ga',
  '2': '5',
  '3': '6',
  '4': '7',
  '5': 'cúig',
  '6': 'seacht',
  'ord': 'séú',
  'opt': 'Personal',
  'o

In [ ]:
collated_templates['acronym']

In [57]:
nonlexical

{'IPA',
 'IPAchar-lite',
 'IPAfont',
 'acronym',
 'adverbial accusative',
 'aii-root',
 'ca-verb-obj',
 'cat',
 'categorize',
 'chess diagram/alt',
 'circa',
 'circa2',
 'coin',
 'coinage',
 'coined',
 'collapse',
 'commonscat',
 'defdate',
 'dercat',
 'es-verb-obj',
 'etydate',
 'etydate/l2',
 'etydate/the',
 'gl',
 'gloss',
 'hanja-dongguk',
 'hanja-hunmong',
 'ic',
 'initialism',
 'it-verb-obj',
 'lang',
 'lit',
 'lw',
 'm-g',
 'named-after',
 'nb...',
 'no deprecated lang param usage',
 'nowrap',
 'nv-link-to-stem',
 'pedia',
 'progreso',
 'ref',
 'refn',
 'ruby',
 'small',
 'topics',
 'uncertain',
 'vern',
 'wp',
 'xlit',
 'zh-ref'}

### Exceptions:

These templates do have their first non-lang key as a word, but we do not detect it:
- acronym (maybe)
- adverbial accusative
- initialism
- cardinalbox: all not words

In [76]:
max([int(k) for k in predicted_word_keys['m']])

2

In [96]:
# these templates take in multiple words
multilexical = {k: v for k, v in predicted_word_keys.items() if len(v) > 1}

# these templates take in arbitrarily many number of words
n_lexical = set()
for tname, keys in predicted_word_keys.items():
    if not keys:
        continue
    try:
        if max([int(k) for k in keys], default=0) >= 6: # if we have a key greater than '6', then it's probably an affix or smth
            n_lexical.add(tname)
        elif '5' in keys and all(key in keys for key in word_success_rates[tname]):
            # all numerical keys from 2-5 are indeed successful
            n_lexical.add(tname)
    except Exception as e:
        print(tname, keys)
        raise e
    

# exceptions
n_lexical.add('Han compound')
n_lexical.add('suf')
n_lexical.add('blend')
n_lexical.add('pre')
n_lexical.remove('cardinalbox')
n_lexical.remove('vi-etym-sino')
n_lexical.remove('t2i-Egyd')

In [95]:
n_lexical

{'Han compound',
 'af',
 'affix',
 'alter',
 'blend',
 'com',
 'com+',
 'com-ja',
 'compound',
 'dbt',
 'doublet',
 'elements',
 'io-bor',
 'ja-com',
 'ja-compound',
 'ko-etym-Sino',
 'ko-etym-sino',
 'ko-sino',
 'nv-prefixes',
 'participle of',
 'pre',
 'prefix',
 'pseudo-loan',
 'root',
 'suf',
 'suffix',
 'surf',
 'surface analysis',
 't2i-Egyd',
 'zh-etym-short',
 'zh-wp'}

In [92]:
predicted_word_keys['blend']

['2', '3', '4']

In [93]:
word_success_rates['blend']

{'2': {'total': 6040, 'success': 5734, 'nonlatin': 622, 'unique': 4173},
 't1': {'total': 653, 'success': 445, 'nonlatin': 19, 'unique': 530},
 '3': {'total': 6023, 'success': 5753, 'nonlatin': 635, 'unique': 4169},
 't2': {'total': 624, 'success': 439, 'nonlatin': 3, 'unique': 524},
 '4': {'total': 174, 'success': 137, 'nonlatin': 79, 'unique': 139},
 'nocap': {'total': 568, 'success': 568, 'nonlatin': 0, 'unique': 6},
 'nocat': {'total': 74, 'success': 74, 'nonlatin': 0, 'unique': 2},
 'tr1': {'total': 156, 'success': 94, 'nonlatin': 63, 'unique': 137},
 'tr2': {'total': 160, 'success': 109, 'nonlatin': 71, 'unique': 142},
 'alt1': {'total': 99, 'success': 20, 'nonlatin': 21, 'unique': 86},
 'alt2': {'total': 75, 'success': 24, 'nonlatin': 23, 'unique': 66},
 'notext': {'total': 52, 'success': 52, 'nonlatin': 0, 'unique': 2},
 'gloss1': {'total': 79, 'success': 46, 'nonlatin': 3, 'unique': 65},
 'gloss2': {'total': 59, 'success': 37, 'nonlatin': 0, 'unique': 55},
 '5': {'total': 33, 

Is correct:
- confix


In [97]:
high_fliers = {k: v for k, v in word_success_rates.items() if '7' in v}

In [ ]:
# for k, v in high_fliers.items():
#     print(k)
# it seems that we do get most of the arbitrarily-many templates

af
affix
prefix
compound
root
com
doublet
ko-etym-sino
vi-etym-sino
blend
com+
ja-compound
dbt
ko-etym-Sino
io-bor
pseudo-loan
nv-prefixes
com-ja
zh-etym-short
zh-wp
t2i-Egyd
